# NB14 — Targeted reruns for reviewer-critical analyses (v2)

**Purpose.** Address four reviewer-critical questions using the existing NB00–NB13 project structure:

1. Compare the prevalence correction with empirical-Bayes (EB) recalibration under the **same local-label budget**, and separately with all labels (explicitly unequal budget).
2. Repeat the hospital-level comparison on hospitals selected **without using 2015 observed deaths**, report it in the same form as Table 5 (DL τ, prediction interval, % in band with the perfect-calibration benchmark, median |ln O/E|, Holm-adjusted Wilcoxon tests against EB), and fit a Poisson–lognormal count model that keeps hospitals with zero observed deaths.
3. Retrain LightGBM using only MIMIC-IV stays whose **latest possible real year is ≤2013**, accounting for MIMIC-IV date shifting (with and without a one-year allowance for the day-level shift), and repeat the target-site comparison.
4. A **hospital-cluster bootstrap** that re-estimates the global calibration step and the network prior in every replicate; DerSimonian–Laird (DL) vs REML priors.

**Changes from v1 (reviewed 19 Oct 2026).**
- Label-budget CIs now resample hospitals and pool all label-subsample repeats, matching the point estimate (v1 drew a single repeat per bootstrap draw, so the interval did not describe the averaged estimate).
- The pipeline bootstrap resamples hospitals only. v1 also resampled patients within each hospital; because the count model already accounts for within-hospital sampling noise, that double-counted noise and inflated τ in every replicate (on synthetic data the bootstrap median τ was ~1.8× the point estimate).
- Outcome-free cohorts are now summarized with the same statistics as Table 5, plus Holm-adjusted Wilcoxon tests and a hospital-flow table.
- Strict pre-2014 cohort: BigQuery client uses Colab auth and the project ID; a conservative variant allows for the ±1-year error that day-level date shifting introduces; pooled performance of the retrained model is reported.
- LightGBM search for the strict model is cached so interrupted runs resume.

**Prerequisites.** NB03, NB04 and NB08 outputs (`data/analysis/*.parquet`, `models/pred_eicu.parquet`, `results/NB03/hospital_year_counts.csv`), and the credentialed PhysioNet BigQuery project for the one metadata query in Section 5. No patient-level data are written to `results/NB14/`.

**Interpretation rules.** `TRANSPORT_FAST=1` is for smoke tests only. The 2015 death count is used as an *outcome* but never as a selection criterion in the outcome-free cohorts. The Poisson–lognormal τ is a different estimator from the DL τ of ln(O/E); compare methods within an estimator, not across estimators.


In [ ]:
# ==== Setup: follow the repository's NB00–NB13 convention ====
import os, sys, json, re, time, math, warnings, subprocess, platform
from pathlib import Path

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ.setdefault('TRANSPORT_BASE', '/content/drive/MyDrive/Plan A/v4')

BASE = os.environ.get('TRANSPORT_BASE', '/content/drive/MyDrive/Plan A/v4')
if not os.path.isdir(BASE):
    raise FileNotFoundError(
        f'TRANSPORT_BASE={BASE!r} does not exist. Set TRANSPORT_BASE to the local project root.'
    )
os.environ['TRANSPORT_BASE'] = BASE
sys.path.insert(0, os.path.join(BASE, 'code'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import transport_utils as tu
from scipy import stats
from scipy.optimize import minimize, minimize_scalar
from scipy.special import gammaln, logsumexp
from numpy.polynomial.hermite import hermgauss
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings('default')
tu.set_style()
OUT = tu.init_notebook('NB14')
SEED = int(tu.SEED)
FAST = bool(tu.FAST)
N_LABEL_REPEATS = 3 if FAST else 50
B_LABEL_CI = 50 if FAST else 500
B_PIPELINE = 10 if FAST else 500
N_SIM_BENCH = 100 if FAST else 1000   # perfect-calibration benchmark simulations
GCP_PROJECT = os.environ.get('GCP_PROJECT', 'mimic-rs')   # same project as NB01/NB02
print({'BASE': BASE, 'FAST': FAST, 'label_repeats': N_LABEL_REPEATS,
       'label_CI_bootstrap': B_LABEL_CI, 'pipeline_bootstrap': B_PIPELINE,
       'benchmark_sims': N_SIM_BENCH, 'gcp_project': GCP_PROJECT})

## 0. Load the original-model predictions and define cohorts

The original 97-hospital set is retained only for a comparable re-analysis of the previously reported label-budget results. It is **not** used as the primary cohort for the outcome-free analysis below.

In [ ]:
m, e = tu.load_analysis()
pred_path = os.path.join(tu.MODELS, 'pred_eicu.parquet')
hy_path = os.path.join(tu.RESULTS, 'NB03', 'hospital_year_counts.csv')
if not os.path.exists(pred_path):
    raise FileNotFoundError(f'Missing {pred_path}; run NB04 first.')
if not os.path.exists(hy_path):
    raise FileNotFoundError(f'Missing {hy_path}; run NB03 first.')

pred = pd.read_parquet(pred_path)
hy = pd.read_csv(hy_path)
required_pred = {'stay_id', 'hospital_id', 'mortality', 'split', 'p_lgbm'}
missing = required_pred - set(pred.columns)
if missing:
    raise ValueError(f'pred_eicu.parquet is missing required columns: {sorted(missing)}')
if pred['stay_id'].duplicated().any():
    raise ValueError('pred_eicu.parquet has duplicate stay_id values; resolve before rerunning.')

upd = pred.loc[pred['split'].eq('update')].copy().reset_index(drop=True)
ev = pred.loc[pred['split'].eq('eval')].copy().reset_index(drop=True)
if upd.empty or ev.empty:
    raise ValueError('Expected both 2014 update and 2015 evaluation rows in pred_eicu.parquet.')

# Per-site arrays and frames; IDs retain the original dtype for joins.
upd_by_h = {h: g.reset_index(drop=True) for h, g in upd.groupby('hospital_id', sort=False)}
ev_by_h = {h: g.reset_index(drop=True) for h, g in ev.groupby('hospital_id', sort=False)}
G = {h: (g['p_lgbm'].to_numpy(float), g['mortality'].to_numpy(float)) for h, g in upd_by_h.items()}
E_ = {h: (g['p_lgbm'].to_numpy(float), g['mortality'].to_numpy(float)) for h, g in ev_by_h.items()}
UPDH = sorted(G)
H_BOTH = sorted(set(upd_by_h).intersection(ev_by_h))
nupd = {h: len(upd_by_h[h]) for h in UPDH}
neval = {h: len(ev_by_h[h]) for h in ev_by_h}
PI_SRC = float(m.loc[m['period'].eq('dev'), 'mortality'].mean())
if not (0 < PI_SRC < 1):
    raise ValueError(f'Invalid source prevalence: {PI_SRC}')

# The original selection used evaluation deaths. Keep it only for matching the existing published tables.
if 'eligible_temporal' not in hy.columns:
    raise ValueError('NB03 hospital_year_counts.csv lacks eligible_temporal; inspect the NB03 output before proceeding.')
eligible_mask = hy['eligible_temporal']
if eligible_mask.dtype == object:
    eligible_mask = eligible_mask.astype(str).str.strip().str.lower().isin({'true', '1', 'yes'})
else:
    eligible_mask = eligible_mask.astype(bool)
ELIG_ORIG = sorted(hy.loc[eligible_mask, 'hospital_id'].tolist())
ELIG_ORIG = [h for h in ELIG_ORIG if h in G and h in E_]
FIX35 = sorted([h for h in ELIG_ORIG if nupd[h] >= 400])

print(f'MIMIC rows={len(m):,}; eICU update rows={len(upd):,}; evaluation rows={len(ev):,}')
print(f'Update hospitals={len(UPDH)}; hospitals in both years={len(H_BOTH)}')
print(f'Original outcome-selected temporal set={len(ELIG_ORIG)}; >=400 update-patient subset={len(FIX35)}')
print(f'Original source prevalence={PI_SRC:.5f}')
if len(ELIG_ORIG) == 0 or len(FIX35) == 0:
    raise ValueError('Original 97/fixed-35 sets could not be reconstructed; inspect NB03 counts and hospital IDs.')

## 1. Statistical helpers

The next cell defines (i) a univariate random-effects REML estimator for the network prior and (ii) a Poisson–lognormal likelihood for site-level observed counts. The latter retains zero-event hospitals because it models observed counts directly instead of taking `log(O/E)` for each hospital.

In [ ]:
def fit_reml_normal_meta(est, se):
    """REML estimate of mu and tau^2 for y_h ~ N(mu, se_h^2 + tau^2)."""
    y = np.asarray(est, dtype=float)
    s = np.asarray(se, dtype=float)
    keep = np.isfinite(y) & np.isfinite(s) & (s > 0)
    y, s = y[keep], s[keep]
    if len(y) < 3:
        return {'k': len(y), 'mu': np.nan, 'tau2': np.nan, 'tau': np.nan, 'converged': False}

    v = s ** 2
    def reml_objective(tau2):
        tau2 = max(float(tau2), 0.0)
        vv = v + tau2
        w = 1.0 / vv
        mu = np.sum(w * y) / np.sum(w)
        resid = y - mu
        # Negative restricted log likelihood up to constants independent of tau^2.
        return 0.5 * (np.log(vv).sum() + np.log(w.sum()) + np.sum(w * resid ** 2))

    upper = max(1.0, float(np.var(y, ddof=1) * 10 + np.max(v) * 10))
    opt = minimize_scalar(reml_objective, bounds=(1e-10, upper), method='bounded',
                          options={'xatol': 1e-10, 'maxiter': 1000})
    tau2 = float(opt.x) if opt.success else 0.0
    # Include the boundary tau^2=0 in the comparison.
    if reml_objective(0.0) <= reml_objective(tau2):
        tau2 = 0.0
    w = 1.0 / (v + tau2)
    mu = float(np.sum(w * y) / np.sum(w))
    return {'k': len(y), 'mu': mu, 'tau2': tau2, 'tau': float(np.sqrt(tau2)),
            'converged': bool(opt.success), 'objective': float(reml_objective(tau2))}


def fit_meta(est, se, method='DL'):
    method = method.upper()
    if method == 'DL':
        r = tu.meta_analysis(est, se, hksj=False)
        return {'k': int(r['k']), 'mu': float(r['mu']), 'tau2': float(r['tau2']),
                'tau': float(r['tau']), 'converged': True, 'method': 'DL'}
    if method == 'REML':
        r = fit_reml_normal_meta(est, se)
        r['method'] = 'REML'
        return r
    raise ValueError("method must be 'DL' or 'REML'")


def fit_poisson_lognormal(O, E, n_quad=30):
    """Fit O_h | u_h ~ Poisson(E_h * exp(mu + u_h)), u_h ~ N(0, tau^2).

    Uses Gauss-Hermite quadrature and maximum marginal likelihood. Sites with O=0 are valid.
    exp(mu) is the median site O/E under the random-effects distribution; exp(mu+tau^2/2)
    is the corresponding model-implied mean multiplier.
    """
    O = np.asarray(O, dtype=float)
    E = np.asarray(E, dtype=float)
    keep = np.isfinite(O) & np.isfinite(E) & (O >= 0) & (E > 0)
    O, E = O[keep], E[keep]
    if len(O) < 3:
        return {'k': int(len(O)), 'mu': np.nan, 'tau': np.nan, 'tau2': np.nan,
                'median_oe_multiplier': np.nan, 'mean_oe_multiplier': np.nan,
                'loglik': np.nan, 'converged': False}

    x, w = hermgauss(n_quad)
    logw = np.log(w) - 0.5 * np.log(np.pi)
    logE = np.log(E)
    log_fact = gammaln(O + 1.0)

    def nll(theta):
        mu, log_tau = float(theta[0]), float(theta[1])
        tau = np.exp(log_tau)
        eta = np.clip(mu + np.sqrt(2.0) * tau * x[None, :], -25.0, 20.0)
        loglam = np.clip(logE[:, None] + eta, -25.0, 25.0)
        lam = np.exp(loglam)
        logpmf = O[:, None] * loglam - lam - log_fact[:, None]
        marginal = logsumexp(logpmf + logw[None, :], axis=1)
        if not np.all(np.isfinite(marginal)):
            return 1e100
        return -float(np.sum(marginal))

    mu0 = float(np.log((O.sum() + 0.5) / E.sum()))
    opt = minimize(nll, x0=np.array([mu0, np.log(0.2)]), method='L-BFGS-B',
                   bounds=[(-8.0, 8.0), (-9.0, 1.5)],
                   options={'maxiter': 2000, 'ftol': 1e-10})
    mu = float(opt.x[0])
    tau = float(np.exp(opt.x[1]))
    return {'k': int(len(O)), 'n_zero_observed': int(np.sum(O == 0)),
            'mu': mu, 'tau': tau, 'tau2': tau * tau,
            'median_oe_multiplier': float(np.exp(mu)),
            'mean_oe_multiplier': float(np.exp(mu + 0.5 * tau * tau)),
            'loglik': float(-opt.fun), 'converged': bool(opt.success),
            'optimizer_message': str(opt.message)}


def clipped_rate(y):
    y = np.asarray(y, dtype=float)
    n = len(y)
    if n == 0:
        raise ValueError('Cannot compute a prevalence correction from zero labels.')
    return float(np.clip(y.mean(), 0.5 / n, 1.0 - 0.5 / n))


def build_G(df, pcol):
    return {h: (g[pcol].to_numpy(float), g['mortality'].to_numpy(float))
            for h, g in df.groupby('hospital_id', sort=False)}


def fit_network_core(G_dict, sites, tau_method='DL'):
    """Global logistic step and plain EB prior from the listed sites' update data."""
    sites = list(sites)
    if len(sites) < 3:
        raise ValueError('At least 3 sites are required to fit a random-effects prior.')
    p_all = np.concatenate([G_dict[h][0] for h in sites])
    y_all = np.concatenate([G_dict[h][1] for h in sites])
    ag, bg = tu.fit_logistic(p_all, y_all)
    if not (np.isfinite(ag) and np.isfinite(bg)):
        raise RuntimeError('Global logistic recalibration failed for this site set.')
    rows = []
    for h in sites:
        p, y = G_dict[h]
        q = tu.apply_logistic(p, ag, bg)
        a, se = tu.fit_intercept(q, y)
        rows.append({'hospital_id': h, 'a': float(a), 'se': float(se), 'n': len(y), 'events': int(y.sum())})
    tab = pd.DataFrame(rows)
    meta = fit_meta(tab['a'], tab['se'], method=tau_method)
    return {'ag': float(ag), 'bg': float(bg), 'mu': float(meta['mu']),
            'tau2': float(meta['tau2']), 'tau': float(meta['tau']),
            'method': tau_method.upper(), 'offsets': tab}


def per_site_counts(frame, pcol, hset):
    q = frame.loc[frame['hospital_id'].isin(hset) & frame[pcol].notna(), ['hospital_id', 'mortality', pcol]]
    return q.groupby('hospital_id').agg(n=('mortality', 'size'), observed=('mortality', 'sum'),
                                        expected=(pcol, 'sum'))


def site_abs_ln_oe(per):
    """|ln O/E| per site; zero-event sites use O=0.5 (as in sitecal.audit) so they are not dropped."""
    return np.abs(np.log(np.maximum(per['observed'], 0.5) / per['expected']))


def perfect_calibration_benchmark(frame, pcol, hset, n_sim, seed):
    """Expected % of sites with O/E in 0.8-1.2 if the predictions were perfectly calibrated
    (outcomes simulated as Bernoulli(p)). A benchmark, not a bound on any updating strategy."""
    rng = np.random.default_rng(seed)
    q = frame.loc[frame['hospital_id'].isin(hset) & frame[pcol].notna()]
    hits, k = np.zeros(n_sim), 0
    for _, g in q.groupby('hospital_id'):
        p = g[pcol].to_numpy(float)
        o_sim = np.zeros(n_sim)
        for s in range(0, len(p), 1000):
            pp = p[s:s + 1000]
            o_sim += (rng.random((n_sim, len(pp))) < pp).sum(axis=1)
        r = o_sim / p.sum()
        hits += (r >= tu.OE_ACCEPT[0]) & (r <= tu.OE_ACCEPT[1])
        k += 1
    return float(100 * hits.mean() / k)


def table5_summary(frame, pcol, hset, n_sim=1000, seed=0):
    """The statistics reported in Table 5, for any hospital set. DL tau uses sites with O>0
    (log O/E undefined otherwise); the count of zero-event sites is reported alongside."""
    per = per_site_counts(frame, pcol, hset)
    pos = per[per['observed'] > 0]
    ln = np.log(pos['observed'] / pos['expected'])
    se = np.sqrt((1 - pos['observed'] / pos['n']) / pos['observed'])
    ma = tu.meta_analysis(ln.to_numpy(), se.to_numpy())
    oe = per['observed'] / per['expected']
    a = site_abs_ln_oe(per)
    return {'sites': int(len(per)), 'sites_zero_observed': int((per['observed'] == 0).sum()),
            'patients': int(per['n'].sum()), 'dl_tau_ln_oe': float(ma['tau']),
            'pi_oe_lo': float(np.exp(ma['pi_lo'])), 'pi_oe_hi': float(np.exp(ma['pi_hi'])),
            're_mean_oe': float(np.exp(ma['mu'])),
            'pct_in_band': float(100 * oe.between(*tu.OE_ACCEPT).mean()),
            'benchmark_pct_in_band': perfect_calibration_benchmark(frame, pcol, hset, n_sim, seed),
            'median_abs_ln_oe': float(a.median()), 'mean_sq_ln_oe': float((a ** 2).mean())}


def wilcoxon_holm_vs_ref(frame, method_cols, ref_label, hset):
    """Paired Wilcoxon signed-rank tests of site |ln O/E|, reference vs each other method, Holm-adjusted."""
    err = {lab: site_abs_ln_oe(per_site_counts(frame, col, hset)) for lab, col in method_cols.items()}
    rows = []
    for lab in method_cols:
        if lab == ref_label:
            continue
        d = pd.concat([err[ref_label].rename('ref'), err[lab].rename('other')], axis=1, join='inner').dropna()
        p = float(stats.wilcoxon(d['ref'], d['other']).pvalue) if len(d) >= 5 and (d['ref'] != d['other']).any() else np.nan
        rows.append({'reference': ref_label, 'comparator': lab, 'sites': len(d),
                     'pct_sites_reference_lower_error': float(100 * (d['ref'] < d['other']).mean()),
                     'p_raw': p})
    out = pd.DataFrame(rows)
    ok = out['p_raw'].notna()
    out['p_holm'] = np.nan
    if ok.any():
        out.loc[ok, 'p_holm'] = multipletests(out.loc[ok, 'p_raw'], method='holm')[1]
    return out

print('Helpers defined. Synthetic smoke test follows.')
# Fast numerical smoke test, not a result from the study data.
_smoke = fit_poisson_lognormal([0, 2, 4, 7], [1.1, 2.2, 3.9, 6.2], n_quad=20)
assert np.isfinite(_smoke['tau']) and _smoke['k'] == 4
print(_smoke)

## 2. Fair prevalence-correction comparison at the same local-label budget

For each hospital and each budget, the same sampled 2014 patient indices are used for local intercept, EB, and prevalence correction. EB's global recalibration and prior are estimated leave-hospital-out from the other hospitals' 2014 data, matching the NB13 label-budget design. The full-label prevalence correction is reported in a separate table and is **not** presented as an equal-budget comparison.

The 97-hospital and fixed-35 sets are retained here to reproduce the original log-O/E summaries, which require positive observed event counts. The new count-model analysis in the next section addresses outcome-free selection and zero-event hospitals.

In [ ]:
# Reconstruct leave-hospital-out plain EB priors. This is cached per hospital so the cell can resume.
loho_cache_path = os.path.join(OUT, '_cache_loho_plain_core.pkl')
if os.path.exists(loho_cache_path):
    LOHO_CORE = pd.read_pickle(loho_cache_path)
    print(f'Loaded LOHO cache: {len(LOHO_CORE)} hospitals')
else:
    LOHO_CORE = {}
    for i, h in enumerate(ELIG_ORIG, start=1):
        others = [x for x in UPDH if x != h]
        LOHO_CORE[h] = fit_network_core(G, others, tau_method='DL')
        if i % 10 == 0 or i == len(ELIG_ORIG):
            pd.to_pickle(LOHO_CORE, loho_cache_path)
            print(f'  LOHO priors {i}/{len(ELIG_ORIG)}', flush=True)

# If resuming from a partial cache, ensure every original comparison site is present.
missing_loho = set(ELIG_ORIG) - set(LOHO_CORE)
if missing_loho:
    for i, h in enumerate(sorted(missing_loho), start=1):
        LOHO_CORE[h] = fit_network_core(G, [x for x in UPDH if x != h], tau_method='DL')
        if i % 10 == 0 or i == len(missing_loho):
            pd.to_pickle(LOHO_CORE, loho_cache_path)

In [ ]:
NS = [25, 50, 100, 200, 400]
lb_rows = []
rng_master = np.random.default_rng(SEED + 1401)

for rep in range(N_LABEL_REPEATS):
    for cohort_name, site_list_fn in [
        ('eligible97_original', lambda n: [h for h in ELIG_ORIG if nupd[h] >= n]),
        ('fixed35_ge400_update', lambda n: list(FIX35)),
    ]:
        for n in NS:
            hs = site_list_fn(n)
            for h in hs:
                pu, yu = G[h]
                pe, ye = E_[h]
                rng = np.random.default_rng(int(rng_master.integers(0, 2**32 - 1)))
                ii = rng.choice(len(yu), size=n, replace=False)
                p_n, y_n = pu[ii], yu[ii]
                rate_n = clipped_rate(y_n)
                # Same local n for all three methods; no site labels from the 2015 outcome are used in fitting.
                p_prev_n = tu.expit(tu.logit(pe) + tu.logit(rate_n) - tu.logit(PI_SRC))
                a_local, _ = tu.fit_intercept(p_n, y_n)
                p_local_n = tu.apply_logistic(pe, a_local)

                F = LOHO_CORE[h]
                q_update = tu.apply_logistic(p_n, F['ag'], F['bg'])
                a_h, se_h = tu.fit_intercept(q_update, y_n)
                b_h = F['tau2'] / (F['tau2'] + se_h ** 2) if F['tau2'] > 0 else 0.0
                a_eb = F['mu'] + b_h * (a_h - F['mu'])
                q_eval = tu.apply_logistic(pe, F['ag'], F['bg'])
                p_eb_n = tu.expit(tu.logit(q_eval) + a_eb)

                for method, p_hat in [
                    ('No updating', pe),
                    ('Local intercept, n labels', p_local_n),
                    ('EB, same n labels (LOHO prior)', p_eb_n),
                    ('Prevalence correction, same n labels', p_prev_n),
                ]:
                    O = float(np.sum(ye)); Ehat = float(np.sum(p_hat))
                    if O <= 0 or Ehat <= 0:
                        continue
                    oe = O / Ehat
                    lb_rows.append({
                        'rep': rep, 'cohort': cohort_name, 'n': n, 'hospital_id': h,
                        'method': method, 'observed': int(O), 'expected': Ehat, 'oe': oe,
                        'abs_ln_oe': abs(float(np.log(oe))), 'sq_ln_oe': float(np.log(oe) ** 2),
                        'in_band': bool(0.8 <= oe <= 1.2)
                    })

lb_long = pd.DataFrame(lb_rows)
if lb_long.empty:
    raise RuntimeError('No equal-budget label results were generated.')
tu.save_table(lb_long, 'A_label_budget_per_hospital', show=False)
print(lb_long.groupby(['cohort', 'n', 'method']).agg(
    hospitals=('hospital_id', 'nunique'), mean_sq_ln_oe=('sq_ln_oe', 'mean'),
    median_abs_ln_oe=('abs_ln_oe', 'median'), pct_in_band=('in_band', lambda x: 100*x.mean())
).round(4).to_string())

In [ ]:
LB_METRICS = ['mean_sq_ln_oe', 'median_abs_ln_oe', 'pct_in_band']
M_NONE, M_LOC = 'No updating', 'Local intercept, n labels'
M_EB, M_PREV = 'EB, same n labels (LOHO prior)', 'Prevalence correction, same n labels'
LB_CONTRASTS = [(M_PREV, M_EB), (M_EB, M_NONE), (M_PREV, M_NONE), (M_LOC, M_EB)]


def _lb_metrics(sq, ab, band):
    return {'mean_sq_ln_oe': float(sq.mean()), 'median_abs_ln_oe': float(np.median(ab)),
            'pct_in_band': float(100 * band.mean())}


def summarize_label_budget(raw, B=500, seed=1410):
    """Point estimates pool all hospitals x label-subsample repeats (as in NB13).
    95% intervals resample hospitals; each resampled hospital brings all of its repeats, so the interval
    describes the same repeat-averaged quantity as the point estimate, with the hospital as the unit."""
    rng = np.random.default_rng(seed)
    rows, paired = [], []
    for (cohort, n), d in raw.groupby(['cohort', 'n'], sort=True):
        methods = sorted(d['method'].unique())
        hids = np.array(sorted(d['hospital_id'].unique()), dtype=object)
        per = {mth: {h: (g['sq_ln_oe'].to_numpy(float), g['abs_ln_oe'].to_numpy(float), g['in_band'].to_numpy(float))
                     for h, g in d[d['method'] == mth].groupby('hospital_id')} for mth in methods}

        def stat(mth, hs):
            parts = [per[mth][h] for h in hs if h in per[mth]]
            return _lb_metrics(*(np.concatenate([p[i] for p in parts]) for i in range(3)))

        point = {mth: stat(mth, hids) for mth in methods}
        draws = {mth: {mt: [] for mt in LB_METRICS} for mth in methods}
        cdraws = {c: {mt: [] for mt in LB_METRICS} for c in LB_CONTRASTS if c[0] in per and c[1] in per}
        for _ in range(B):
            hs = rng.choice(hids, size=len(hids), replace=True)
            vals = {mth: stat(mth, hs) for mth in methods}
            for mth in methods:
                for mt in LB_METRICS:
                    draws[mth][mt].append(vals[mth][mt])
            for (a, b) in cdraws:
                for mt in LB_METRICS:
                    cdraws[(a, b)][mt].append(vals[a][mt] - vals[b][mt])
        for mth in methods:
            row = {'cohort': cohort, 'n': n, 'method': mth, 'hospitals': len(hids), **point[mth]}
            for mt in LB_METRICS:
                row[mt + '_lo'], row[mt + '_hi'] = np.percentile(draws[mth][mt], [2.5, 97.5])
            rows.append(row)
        for (a, b), dd in cdraws.items():
            for mt in LB_METRICS:
                lo, hi = np.percentile(dd[mt], [2.5, 97.5])
                paired.append({'cohort': cohort, 'n': n, 'contrast': f'{a} minus {b}', 'metric': mt,
                               'estimate': point[a][mt] - point[b][mt], 'ci_lo': lo, 'ci_hi': hi,
                               'hospitals': len(hids), 'bootstrap_replicates': B})
    return pd.DataFrame(rows), pd.DataFrame(paired)


lb_summary, lb_paired = summarize_label_budget(lb_long, B=B_LABEL_CI)
tu.save_table(lb_summary.round(5), 'A_label_budget_summary')
tu.save_table(lb_paired.round(5), 'A_label_budget_paired_contrasts')
print('Sign convention: for mean_sq / median_abs a negative estimate favours the first method; '
      'for pct_in_band a positive estimate favours the first method.')


In [ ]:
# Full-label prevalence correction: explicitly unequal local-label budget; separate from Section A's matched-n comparison.
prev_full_rows = []
for cohort_name, hlist in [('eligible97_original', ELIG_ORIG), ('fixed35_ge400_update', FIX35)]:
    for h in hlist:
        pu, yu = G[h]
        pe, ye = E_[h]
        rate_all = clipped_rate(yu)
        p_prev_all = tu.expit(tu.logit(pe) + tu.logit(rate_all) - tu.logit(PI_SRC))
        O, Ehat = float(ye.sum()), float(p_prev_all.sum())
        if O <= 0 or Ehat <= 0:
            continue
        oe = O / Ehat
        prev_full_rows.append({'cohort': cohort_name, 'hospital_id': h, 'n_update_labels_used': len(yu),
                               'method': 'Prevalence correction, all 2014 labels (unequal-budget comparator)',
                               'observed': int(O), 'expected': Ehat, 'oe': oe,
                               'abs_ln_oe': abs(np.log(oe)), 'sq_ln_oe': np.log(oe)**2,
                               'in_band': bool(0.8 <= oe <= 1.2)})
prev_full = pd.DataFrame(prev_full_rows)
tu.save_table(prev_full, 'A_prevalence_full_labels_per_hospital', show=False)
prev_full_summary = prev_full.groupby('cohort').agg(
    hospitals=('hospital_id', 'nunique'), mean_sq_ln_oe=('sq_ln_oe', 'mean'),
    median_abs_ln_oe=('abs_ln_oe', 'median'), pct_in_band=('in_band', lambda x: 100*x.mean()),
    mean_local_labels=('n_update_labels_used', 'mean')
).reset_index().round(5)
tu.save_table(prev_full_summary, 'A_prevalence_full_labels_summary')

## 3. Outcome-free hospital selection and Poisson–lognormal count-model sensitivity

The **all-both-years cohort** includes all hospitals with 2014 update data, 2015 evaluation data, and positive model-expected deaths; this is the primary count-model sensitivity and includes hospitals with zero observed 2015 deaths. A second cohort approximates the original volume requirements but replaces the 2015 observed-death threshold with **expected deaths under the unadjusted model**. The original event-selected 97-hospital cohort is reported only as a comparison row.

In [ ]:
# Fit the original full-network updating strategies for the point estimates, using the shared implementation.
point_pred, original_info, original_eb_table = tu.run_strategies(upd, ev, pcol='p_lgbm', ycol='mortality')
point_pred = point_pred.copy()

# Add the original prevalence correction using all 2014 labels (matching the currently reported baseline).
rate_map_full = {h: clipped_rate(g['mortality'].to_numpy(float)) for h, g in upd_by_h.items()}
rate_eval = point_pred['hospital_id'].map(rate_map_full).astype(float).to_numpy()
point_pred['S6_prev_full_labels'] = tu.expit(
    tu.logit(point_pred['p_lgbm'].to_numpy(float)) + tu.logit(rate_eval) - tu.logit(PI_SRC)
)

# Also derive EB predictions with a REML-fitted network prior for the estimator sensitivity.
NET_REML_POINT = fit_network_core(G, UPDH, tau_method='REML')
off_reml = NET_REML_POINT['offsets'].set_index('hospital_id')
site_reml = off_reml['a'].to_dict()
se_reml = off_reml['se'].to_dict()
for h in point_pred['hospital_id'].unique():
    if h not in site_reml:
        continue
    mask = point_pred['hospital_id'].eq(h).to_numpy()
    a, se = site_reml[h], se_reml[h]
    b = NET_REML_POINT['tau2'] / (NET_REML_POINT['tau2'] + se**2) if NET_REML_POINT['tau2'] > 0 else 0.0
    a_star = NET_REML_POINT['mu'] + b * (a - NET_REML_POINT['mu'])
    q = tu.apply_logistic(point_pred.loc[mask, 'p_lgbm'].to_numpy(float), NET_REML_POINT['ag'], NET_REML_POINT['bg'])
    point_pred.loc[mask, 'S3_eb_REML'] = tu.expit(tu.logit(q) + a_star)

METHOD_COLS = {
    'No updating': 'S0_none',
    'Global logistic': 'S1b_global_logistic',
    'Local intercept': 'S2a_local_intercept',
    'Empirical Bayes (DL prior)': 'S3_eb',
    'Empirical Bayes (REML prior)': 'S3_eb_REML',
    'Prevalence correction (all labels)': 'S6_prev_full_labels',
}
for label, col in METHOD_COLS.items():
    if col not in point_pred.columns:
        raise ValueError(f'Missing method prediction column {col}; check the shared transport_utils version.')

# Full-label prevalence correction vs full-label EB, with a paired site bootstrap.
# Intervals are conditional on fitted predictions/updates; Section 4 re-estimates the network prior.
def _per_site_log_oe(frame, pcol, hset):
    q = frame.loc[frame['hospital_id'].isin(hset) & frame[pcol].notna()].copy()
    z = q.groupby('hospital_id').agg(observed=('mortality', 'sum'), expected=(pcol, 'sum')).reset_index()
    z = z.loc[(z['observed'] > 0) & (z['expected'] > 0)].copy()
    z['abs_ln_oe'] = np.abs(np.log(z['observed'] / z['expected']))
    z['sq_ln_oe'] = np.log(z['observed'] / z['expected']) ** 2
    z['in_band'] = (z['observed'] / z['expected']).between(0.8, 1.2)
    return z.set_index('hospital_id').sort_index()

def _stat_summary(z, metric):
    if metric == 'mean_sq_ln_oe': return float(z['sq_ln_oe'].mean())
    if metric == 'median_abs_ln_oe': return float(z['abs_ln_oe'].median())
    if metric == 'pct_in_band': return float(100 * z['in_band'].mean())
    raise ValueError(metric)

full_label_boot_rows = []
full_label_paired_rows = []
rng_full_boot = np.random.default_rng(SEED + 1414)
for cohort_name, cohort_sites in [('eligible97_original', ELIG_ORIG), ('fixed35_ge400_update', FIX35)]:
    zz = {
        'Empirical Bayes (all labels)': _per_site_log_oe(point_pred, 'S3_eb', cohort_sites),
        'Prevalence correction (all labels)': _per_site_log_oe(point_pred, 'S6_prev_full_labels', cohort_sites),
    }
    common = sorted(set(zz['Empirical Bayes (all labels)'].index) & set(zz['Prevalence correction (all labels)'].index))
    if len(common) < 3:
        raise RuntimeError(f'Too few nonzero-event sites for full-label paired comparison: {cohort_name}')
    metrics = ['mean_sq_ln_oe', 'median_abs_ln_oe', 'pct_in_band']
    site_ids = np.asarray(common, dtype=object)
    point_summaries = {method: {metric: _stat_summary(zz[method].loc[common], metric) for metric in metrics}
                       for method in zz}
    site_draws = {method: {metric: [] for metric in metrics} for method in zz}
    diff_draws = {metric: [] for metric in metrics}
    for _b in range(B_LABEL_CI):
        sampled = rng_full_boot.choice(site_ids, size=len(site_ids), replace=True)
        for method in zz:
            zboot = zz[method].loc[list(sampled)]
            for metric in metrics:
                site_draws[method][metric].append(_stat_summary(zboot, metric))
        for metric in metrics:
            diff_draws[metric].append(
                _stat_summary(zz['Prevalence correction (all labels)'].loc[list(sampled)], metric) -
                _stat_summary(zz['Empirical Bayes (all labels)'].loc[list(sampled)], metric))
    for method in zz:
        for metric in metrics:
            arr = np.asarray(site_draws[method][metric], float)
            full_label_boot_rows.append({'cohort': cohort_name, 'comparison_type': 'method_summary', 'method': method,
                'metric': metric, 'estimate': point_summaries[method][metric],
                'ci_low': float(np.quantile(arr, .025)), 'ci_high': float(np.quantile(arr, .975)),
                'n_sites': len(common), 'bootstrap_replicates': B_LABEL_CI,
                'interval_note': 'Site bootstrap conditional on fitted updates/predictions.'})
    for metric in metrics:
        arr = np.asarray(diff_draws[metric], float)
        full_label_paired_rows.append({'cohort': cohort_name, 'contrast': 'prevalence_all_labels_minus_EB_all_labels',
            'metric': metric, 'estimate': point_summaries['Prevalence correction (all labels)'][metric] -
                point_summaries['Empirical Bayes (all labels)'][metric],
            'ci_low': float(np.quantile(arr, .025)), 'ci_high': float(np.quantile(arr, .975)),
            'n_sites': len(common), 'bootstrap_replicates': B_LABEL_CI,
            'interpretation': 'Negative favors prevalence correction for error metrics; positive favors prevalence correction for pct_in_band.'})
tu.save_table(pd.DataFrame(full_label_boot_rows).round(6), 'A_full_label_method_summary_bootstrap')
tu.save_table(pd.DataFrame(full_label_paired_rows).round(6), 'A_full_label_paired_contrasts_bootstrap')

# Site-level selection inputs use counts and expected deaths only; deaths_eval is stored for audit, not used in selection.
site_rows = []
for h in H_BOTH:
    ug = upd_by_h[h]
    eg = ev_by_h[h]
    e_base = float(eg['p_lgbm'].sum())
    site_rows.append({'hospital_id': h, 'n_update_2014': len(ug), 'n_eval_2015': len(eg),
                      'observed_deaths_2015_audit_only': int(eg['mortality'].sum()),
                      'expected_deaths_2015_unadjusted': e_base,
                      'meets_update_n50': len(ug) >= 50,
                      'meets_eval_n100': len(eg) >= 100,
                      'meets_expected_deaths_10': e_base >= 10.0})
site_selection = pd.DataFrame(site_rows)
site_selection['cohort_all_both_years'] = (site_selection['n_update_2014'] >= 1) & \
    (site_selection['n_eval_2015'] >= 1) & (site_selection['expected_deaths_2015_unadjusted'] > 0)
site_selection['cohort_outcome_free_volume_expected'] = site_selection['meets_update_n50'] & \
    site_selection['meets_eval_n100'] & site_selection['meets_expected_deaths_10']
site_selection['cohort_original_outcome_selected_97'] = site_selection['hospital_id'].isin(ELIG_ORIG)
H_ALL_BOTH = sorted(site_selection.loc[site_selection['cohort_all_both_years'], 'hospital_id'].tolist())
H_STRICT = sorted(site_selection.loc[site_selection['cohort_outcome_free_volume_expected'], 'hospital_id'].tolist())
if len(H_ALL_BOTH) < 3 or len(H_STRICT) < 3:
    raise ValueError('Too few hospitals in the outcome-free cohorts; inspect the selection table before fitting the count model.')
tu.save_table(site_selection.sort_values('hospital_id'), 'B_outcome_free_selection_audit', show=False)
print('Outcome-free all-both-years cohort:', len(H_ALL_BOTH))
print('Outcome-free volume/expected-event cohort:', len(H_STRICT))
print('Original event-selected cohort (for comparison only):', len(ELIG_ORIG))
# Structural check: the primary cohort flag is exactly the outcome-free selection rule.
expected_selection = (site_selection['meets_update_n50'] & site_selection['meets_eval_n100'] &
                      site_selection['meets_expected_deaths_10'])
assert site_selection['cohort_outcome_free_volume_expected'].equals(expected_selection)

# Hospital-flow table (Reviewer 2, item 3): denominators of every analysis in this notebook.
flow = pd.DataFrame([
    {'set': 'eICU hospitals with 2014 update data', 'hospitals': len(UPDH), 'uses_2015_deaths_for_selection': False},
    {'set': 'eICU hospitals with 2015 evaluation data', 'hospitals': len(E_), 'uses_2015_deaths_for_selection': False},
    {'set': 'Both years (count-model cohort)', 'hospitals': len(H_ALL_BOTH), 'uses_2015_deaths_for_selection': False},
    {'set': 'Outcome-free: >=50 update, >=100 eval patients, >=10 expected deaths', 'hospitals': len(H_STRICT),
     'uses_2015_deaths_for_selection': False},
    {'set': 'Original: >=50 update, >=100 eval patients, >=10 observed deaths (Table 5)', 'hospitals': len(ELIG_ORIG),
     'uses_2015_deaths_for_selection': True},
    {'set': 'In both the outcome-free and the original set', 'hospitals': len(set(H_STRICT) & set(ELIG_ORIG)),
     'uses_2015_deaths_for_selection': None},
    {'set': 'Original >=400 update patients (fixed-35 set)', 'hospitals': len(FIX35), 'uses_2015_deaths_for_selection': True},
])
flow['eval_patients_2015'] = [len(ev), len(ev), int(ev['hospital_id'].isin(H_ALL_BOTH).sum()),
                              int(ev['hospital_id'].isin(H_STRICT).sum()), int(ev['hospital_id'].isin(ELIG_ORIG).sum()),
                              int(ev['hospital_id'].isin(set(H_STRICT) & set(ELIG_ORIG)).sum()),
                              int(ev['hospital_id'].isin(FIX35).sum())]
tu.save_table(flow, 'B_hospital_flow')


In [ ]:
# Use the count likelihood to retain zero-event hospitals. This is a sensitivity model, distinct from DL meta-analysis of log(O/E).
# Keep patient-level predictions in memory only; do not place them in the result archive.
cohorts = {
    'all_both_years_outcome_free': H_ALL_BOTH,
    'outcome_free_volume_expected': H_STRICT,
    'original_event_selected_97_comparison_only': [h for h in ELIG_ORIG if h in H_BOTH],
}
site_count_rows, pln_rows = [], []
for cohort_name, hset in cohorts.items():
    for label, pcol in METHOD_COLS.items():
        q = point_pred.loc[point_pred['hospital_id'].isin(hset)].copy()
        if q[pcol].isna().any():
            print(f'Warning: dropping missing predictions for {label} in {cohort_name}')
            q = q.loc[q[pcol].notna()]
        per_h = q.groupby('hospital_id').agg(
            n=('mortality', 'size'), observed=('mortality', 'sum'), expected=(pcol, 'sum')
        ).reset_index()
        per_h['oe'] = per_h['observed'] / per_h['expected']
        per_h['cohort'] = cohort_name
        per_h['method'] = label
        per_h['selection_uses_2015_observed_deaths'] = cohort_name == 'original_event_selected_97_comparison_only'
        site_count_rows.extend(per_h.to_dict('records'))
        fit = fit_poisson_lognormal(per_h['observed'].to_numpy(float), per_h['expected'].to_numpy(float))
        fit.update({'cohort': cohort_name, 'method': label,
                    'selection_uses_2015_observed_deaths': cohort_name == 'original_event_selected_97_comparison_only'})
        pln_rows.append(fit)

site_counts_pln = pd.DataFrame(site_count_rows)
pln_summary = pd.DataFrame(pln_rows)
tu.save_table(site_counts_pln, 'B_site_counts_O_E_all_methods', show=False)
tu.save_table(pln_summary.round(6), 'B_poisson_lognormal_summary')
print(pln_summary[['cohort', 'method', 'k', 'n_zero_observed', 'tau', 'median_oe_multiplier', 'mean_oe_multiplier', 'converged']].round(4).to_string(index=False))

# Same statistics as Table 5 (log O/E meta-analysis, band with benchmark, median), per cohort and method.
t5_rows = []
for cohort_name, hset in cohorts.items():
    for label, pcol in METHOD_COLS.items():
        r = table5_summary(point_pred, pcol, hset, n_sim=N_SIM_BENCH, seed=SEED)
        t5_rows.append({'cohort': cohort_name, 'method': label, **r})
t5 = pd.DataFrame(t5_rows)
tu.save_table(t5.round(4), 'B_table5_style_by_cohort')

wil = pd.concat([wilcoxon_holm_vs_ref(point_pred, METHOD_COLS, 'Empirical Bayes (DL prior)', hset).assign(cohort=c)
                 for c, hset in cohorts.items()], ignore_index=True)
tu.save_table(wil.round(5), 'B_wilcoxon_holm_vs_EB')


## 4. Hospital-cluster bootstrap that re-estimates the global step and network prior

Each replicate resamples the 2014 update hospitals with replacement and keeps each sampled hospital's data intact (its 2014 stays for the update and its 2015 stays for evaluation). The global logistic recalibration, the site offsets and the DL and REML network priors are refitted in every replicate, so the intervals include the uncertainty of the prior. Patients are **not** resampled within hospitals: the count model already accounts for within-hospital sampling noise, and resampling patients as well would count that noise twice and inflate τ.

Evaluation uses the outcome-free cohort (`H_STRICT`). Results are checkpointed. Run with `TRANSPORT_FAST=0` for paper-facing results.


In [ ]:
def fit_network_from_site_arrays(site_arrays):
    """Global logistic step and DL/REML priors from a bootstrap list of (p, y) arrays.
    Repeated draws of a hospital are treated as distinct pseudo-sites."""
    p_all = np.concatenate([z[0] for z in site_arrays])
    y_all = np.concatenate([z[1] for z in site_arrays])
    ag, bg = tu.fit_logistic(p_all, y_all)
    if not (np.isfinite(ag) and np.isfinite(bg)):
        raise RuntimeError('Global logistic step did not converge in a bootstrap replicate.')
    infos = []
    for p, y in site_arrays:
        a_raw, _ = tu.fit_intercept(p, y)
        a_g, se_g = tu.fit_intercept(tu.apply_logistic(p, ag, bg), y)
        infos.append({'a_raw': float(a_raw), 'a_g': float(a_g), 'se_g': float(se_g), 'rate': clipped_rate(y)})
    a = np.array([r['a_g'] for r in infos]); se = np.array([r['se_g'] for r in infos])
    return {'ag': float(ag), 'bg': float(bg), 'infos': infos, 'DL': fit_meta(a, se, 'DL'), 'REML': fit_meta(a, se, 'REML')}


def _dl_tau_from_counts(O, Ehat, n):
    O, Ehat, n = map(np.asarray, (O, Ehat, n))
    k = O > 0
    se = np.sqrt((1 - O[k] / n[k]) / O[k])
    return float(tu.meta_analysis(np.log(O[k] / Ehat[k]), se, hksj=False)['tau'])


def run_pipeline_bootstrap_rep(rep, target_set, n_quad=20):
    rng = np.random.default_rng(SEED + 145000 + int(rep))
    draw_h = rng.choice(np.asarray(UPDH, dtype=object), size=len(UPDH), replace=True)
    net = fit_network_from_site_arrays([G[h] for h in draw_h])
    results = {k: [] for k in METHOD_COLS}
    for pseudo_id, h in enumerate(draw_h):
        if h not in target_set:
            continue
        pe, ye = E_[h]
        info = net['infos'][pseudo_id]
        q_eval = tu.apply_logistic(pe, net['ag'], net['bg'])
        O, n = float(ye.sum()), len(ye)
        for est_name in ('DL', 'REML'):
            meta = net[est_name]
            tau2 = float(meta['tau2'])
            B_h = tau2 / (tau2 + info['se_g'] ** 2) if tau2 > 0 else 0.0
            a_star = float(meta['mu'] + B_h * (info['a_g'] - meta['mu']))
            results[f'Empirical Bayes ({est_name} prior)'].append((O, float(tu.expit(tu.logit(q_eval) + a_star).sum()), n))
        results['No updating'].append((O, float(pe.sum()), n))
        results['Global logistic'].append((O, float(q_eval.sum()), n))
        results['Local intercept'].append((O, float(tu.apply_logistic(pe, info['a_raw']).sum()), n))
        results['Prevalence correction (all labels)'].append(
            (O, float(tu.expit(tu.logit(pe) + tu.logit(info['rate']) - tu.logit(PI_SRC)).sum()), n))
    out = []
    for method, trip in results.items():
        if len(trip) < 3:
            continue
        O = np.array([t[0] for t in trip]); Eh = np.array([t[1] for t in trip]); nn = np.array([t[2] for t in trip])
        fit = fit_poisson_lognormal(O, Eh, n_quad=n_quad)
        out.append({'rep': int(rep), 'method': method, **fit, 'dl_tau_ln_oe': _dl_tau_from_counts(O, Eh, nn),
                    'network_mu_DL': net['DL']['mu'], 'network_tau_DL': net['DL']['tau'],
                    'network_mu_REML': net['REML']['mu'], 'network_tau_REML': net['REML']['tau'],
                    'n_eval_sites_bootstrap': len(trip)})
    return out


BOOT_PATH = os.path.join(OUT, 'C_pipeline_bootstrap_partial_v2.csv')   # v2: v1 checkpoints are not reused
if os.path.exists(BOOT_PATH):
    boot_existing = pd.read_csv(BOOT_PATH)
    completed_reps = set(boot_existing['rep'].astype(int).unique())
    boot_rows = boot_existing.to_dict('records')
    print(f'Resuming pipeline bootstrap: {len(completed_reps)} completed replicates')
else:
    completed_reps, boot_rows = set(), []

H_STRICT_SET = set(H_STRICT)
for rep in range(B_PIPELINE):
    if rep in completed_reps:
        continue
    boot_rows.extend(run_pipeline_bootstrap_rep(rep, H_STRICT_SET))
    completed_reps.add(rep)
    if (rep + 1) % 10 == 0 or rep == B_PIPELINE - 1:
        pd.DataFrame(boot_rows).to_csv(BOOT_PATH, index=False)
        print(f'  pipeline bootstrap {rep + 1}/{B_PIPELINE} done', flush=True)

boot_df = pd.DataFrame(boot_rows).drop_duplicates(['rep', 'method'], keep='last')
tu.save_table(boot_df, 'C_pipeline_bootstrap_draws', show=False)


In [ ]:
# Point estimates on the original data (same cohort and estimators) for reference beside the bootstrap.
point_ref = {}
for label, pcol in METHOD_COLS.items():
    per = per_site_counts(point_pred, pcol, H_STRICT)
    point_ref[label] = {'pln_tau_point': fit_poisson_lognormal(per['observed'], per['expected'])['tau'],
                        'dl_tau_point': _dl_tau_from_counts(per['observed'], per['expected'], per['n'])}

boot_summary_rows = []
for method, g in boot_df.groupby('method'):
    boot_summary_rows.append({'method': method, 'n_replicates': g['rep'].nunique(),
        'pln_tau_point': point_ref[method]['pln_tau_point'],
        'pln_tau_lo': float(g['tau'].quantile(0.025)), 'pln_tau_hi': float(g['tau'].quantile(0.975)),
        'pln_tau_boot_median': float(g['tau'].median()),
        'dl_tau_point': point_ref[method]['dl_tau_point'],
        'dl_tau_lo': float(g['dl_tau_ln_oe'].quantile(0.025)), 'dl_tau_hi': float(g['dl_tau_ln_oe'].quantile(0.975)),
        'share_not_converged': float(1 - g['converged'].astype(bool).mean()),
        'median_eval_sites_per_replicate': float(g['n_eval_sites_bootstrap'].median())})
boot_summary = pd.DataFrame(boot_summary_rows)
tu.save_table(boot_summary.round(5), 'C_pipeline_bootstrap_summary')

paired_boot_rows = []
ref = 'Empirical Bayes (DL prior)'
for col, lab in [('tau', 'Poisson-lognormal tau'), ('dl_tau_ln_oe', 'DL tau of ln O/E')]:
    wide = boot_df.pivot(index='rep', columns='method', values=col)
    for method in wide.columns:
        if method == ref:
            continue
        d = (wide[method] - wide[ref]).dropna()
        paired_boot_rows.append({'estimator': lab, 'comparison': f'{method} minus {ref}', 'n_replicates': len(d),
            'point_delta': point_ref[method]['pln_tau_point' if col == 'tau' else 'dl_tau_point']
                           - point_ref[ref]['pln_tau_point' if col == 'tau' else 'dl_tau_point'],
            'delta_lo': float(d.quantile(0.025)), 'delta_hi': float(d.quantile(0.975)),
            'share_replicates_EB_lower': float((d > 0).mean())})
tu.save_table(pd.DataFrame(paired_boot_rows).round(5), 'C_pipeline_bootstrap_paired_tau')

NET_DL_POINT = fit_network_core(G, UPDH, tau_method='DL')
prior_point = pd.DataFrame([
    {'estimator': 'DL', 'mu': NET_DL_POINT['mu'], 'tau2': NET_DL_POINT['tau2'], 'tau': NET_DL_POINT['tau'],
     'boot_tau_lo': float(boot_df['network_tau_DL'].quantile(0.025)), 'boot_tau_hi': float(boot_df['network_tau_DL'].quantile(0.975))},
    {'estimator': 'REML', 'mu': NET_REML_POINT['mu'], 'tau2': NET_REML_POINT['tau2'], 'tau': NET_REML_POINT['tau'],
     'boot_tau_lo': float(boot_df['network_tau_REML'].quantile(0.025)), 'boot_tau_hi': float(boot_df['network_tau_REML'].quantile(0.975))},
])
tu.save_table(prior_point.round(6), 'C_network_prior_DL_vs_REML_point_estimates')


## 5. Retrain LightGBM using only MIMIC-IV stays whose latest possible real year is ≤2013

MIMIC-IV shifts each patient's dates by a fixed number of days. `anchor_year_group` gives the real 3-year range of the patient's `anchor_year` (a shifted year). For each stay the real year is bounded by `anchor_year_group` plus the gap between the stay's shifted year and `anchor_year`. Because the shift is a number of days, not whole years, that gap can be off by one year, so the bound is computed twice:

- `real_year_max` (no allowance) and
- `real_year_max_conservative = real_year_max + 1` (allows for the day-level shift).

Two models are trained: **strict** (conservative bound ≤2013, primary for the claim "trained only on stays before 2014") and **approx** (bound ≤2013 without the allowance). Only stay ID and date-shift metadata are queried from BigQuery; no clinical data are downloaded in this step.


In [ ]:
TIME_META_PATH = os.path.join(tu.DATA_RAW, 'mimic_stay_time_metadata.parquet')
needed_time_cols = {'stay_id', 'shifted_stay_year', 'anchor_year', 'anchor_year_group'}
if needed_time_cols.issubset(m.columns):
    time_meta = m[list(needed_time_cols)].drop_duplicates('stay_id').copy()
elif os.path.exists(TIME_META_PATH):
    time_meta = pd.read_parquet(TIME_META_PATH)
    missing = needed_time_cols - set(time_meta.columns)
    if missing:
        raise ValueError(f'Cached date-shift metadata is missing columns: {sorted(missing)}')
else:
    from google.cloud import bigquery
    if 'google.colab' in sys.modules:
        from google.colab import auth
        auth.authenticate_user()
    sql = """
    SELECT i.stay_id AS stay_id, EXTRACT(YEAR FROM i.intime) AS shifted_stay_year,
           p.anchor_year AS anchor_year, p.anchor_year_group AS anchor_year_group
    FROM `physionet-data.mimiciv_3_1_icu.icustays` AS i
    INNER JOIN `physionet-data.mimiciv_3_1_hosp.patients` AS p USING (subject_id)
    """
    time_meta = bigquery.Client(project=GCP_PROJECT).query(sql).to_dataframe(create_bqstorage_client=False)
    time_meta['stay_id'] = pd.to_numeric(time_meta['stay_id'], errors='raise').astype(m['stay_id'].dtype)
    time_meta.to_parquet(TIME_META_PATH, index=False)

time_meta = time_meta[['stay_id', 'shifted_stay_year', 'anchor_year', 'anchor_year_group']].drop_duplicates('stay_id').copy()
for c_ in ['shifted_stay_year', 'anchor_year']:
    time_meta[c_] = pd.to_numeric(time_meta[c_], errors='coerce')
yrs = time_meta['anchor_year_group'].astype(str).str.extract(r'(\d{4})\s*-\s*(\d{4})')
time_meta['anchor_group_start'] = pd.to_numeric(yrs[0], errors='coerce')
time_meta['anchor_group_end'] = pd.to_numeric(yrs[1], errors='coerce')
bad = time_meta[['shifted_stay_year', 'anchor_year', 'anchor_group_start', 'anchor_group_end']].isna().any(axis=1)
if bad.any():
    raise ValueError(f'{int(bad.sum())} MIMIC stays lack usable anchor/date-shift metadata.')
time_meta['shifted_year_gap'] = time_meta['shifted_stay_year'] - time_meta['anchor_year']
time_meta['real_year_min'] = time_meta['anchor_group_start'] + time_meta['shifted_year_gap']
time_meta['real_year_max'] = time_meta['anchor_group_end'] + time_meta['shifted_year_gap']
time_meta['real_year_max_conservative'] = time_meta['real_year_max'] + 1

m_time = m.drop(columns=[c_ for c_ in ['anchor_year', 'shifted_stay_year'] if c_ in m.columns])
m_time = m_time.merge(time_meta[['stay_id', 'shifted_stay_year', 'anchor_year', 'shifted_year_gap',
                                 'real_year_min', 'real_year_max', 'real_year_max_conservative']],
                      on='stay_id', how='left', validate='one_to_one')
if m_time['real_year_max'].isna().any():
    raise ValueError(f'{int(m_time["real_year_max"].isna().sum())} analysis stays did not join to date metadata.')

PRE_VARIANTS = {'strict': m_time['real_year_max_conservative'] <= 2013,
                'approx': m_time['real_year_max'] <= 2013}
M_PRE = {k: m_time.loc[mask].reset_index(drop=True) for k, mask in PRE_VARIANTS.items()}
for k, d in M_PRE.items():
    if len(d) < 1000 or d['mortality'].sum() < 100:
        raise RuntimeError(f'Pre-2014 cohort {k!r} is too small ({len(d)} stays, {int(d["mortality"].sum())} deaths).')

calendar_audit = m_time.groupby('anchor_year_group').agg(
    analysis_stays=('stay_id', 'size'),
    latest_possible_year_median=('real_year_max', 'median'),
    retained_approx=('real_year_max', lambda s: int((s <= 2013).sum())),
    retained_strict=('real_year_max_conservative', lambda s: int((s <= 2013).sum()))).reset_index()
tu.save_table(calendar_audit, 'D_mimic_date_shift_year_range_audit')
tu.save_table(pd.DataFrame([{'variant': k, 'analysis_stays_original': len(m), 'stays_retained': len(d),
                             'retained_fraction': len(d) / len(m), 'deaths_retained': int(d['mortality'].sum()),
                             'mortality_retained': float(d['mortality'].mean()),
                             'rule': 'real_year_max+1 <= 2013' if k == 'strict' else 'real_year_max <= 2013'}
                            for k, d in M_PRE.items()]), 'D_pre2014_cohort_summary')


### 5.1 Retune and retrain LightGBM within each earlier training window

The NB04 search is repeated inside each window (same search space, folds and early stopping as NB04), so no tuning information comes from post-2013 stays. Searches are cached.


In [ ]:
import joblib
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, ParameterSampler
from sklearn.metrics import log_loss

FEATS = tu.feature_set('V1')
SPACE = dict(num_leaves=[15, 31, 63, 127], learning_rate=[0.02, 0.05, 0.1],
             min_child_samples=[20, 50, 100, 200], subsample=[0.7, 0.85, 1.0],
             colsample_bytree=[0.5, 0.7, 0.9], reg_lambda=[0, 1, 5, 10], max_depth=[-1, 6, 10])
N_SEARCH = 4 if FAST else 25


def search_window(X, y):
    folds = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(X, y))
    rows = []
    for i, params in enumerate(ParameterSampler(SPACE, N_SEARCH, random_state=SEED), start=1):
        losses, iters = [], []
        for tr, va in folds:
            mdl = lgb.LGBMClassifier(n_estimators=3000, subsample_freq=1, random_state=SEED, verbose=-1, n_jobs=-1, **params)
            mdl.fit(X.iloc[tr], y[tr], eval_set=[(X.iloc[va], y[va])], eval_metric='binary_logloss',
                    callbacks=[lgb.early_stopping(100, verbose=False)])
            losses.append(log_loss(y[va], mdl.predict_proba(X.iloc[va], num_iteration=mdl.best_iteration_)[:, 1]))
            iters.append(int(mdl.best_iteration_ or mdl.n_estimators))
        rows.append(dict(params, cv_logloss=float(np.mean(losses)), n_estimators=int(np.mean(iters))))
        print(f'    config {i}/{N_SEARCH}: logloss={rows[-1]["cv_logloss"]:.5f}', flush=True)
    return pd.DataFrame(rows).sort_values('cv_logloss').reset_index(drop=True)


tt_m = m[m['period'] == 'temporal_test'].reset_index(drop=True)
PRED_PRE, PI_SRC_PRE, perf_rows = {}, {}, []
for variant, d in M_PRE.items():
    print(f'[{variant}] training rows={len(d):,}, deaths={int(d["mortality"].sum()):,}')
    X_pre, y_pre = d[FEATS].astype(float), d['mortality'].astype(int).to_numpy()
    srch = tu.cached(f'pre2014_{variant}_search', search_window, X_pre, y_pre)
    tu.save_table(srch, f'D_pre2014_{variant}_lgbm_search', show=False)
    best = srch.iloc[0].to_dict()
    params = {k: (int(best[k]) if k in ('num_leaves', 'min_child_samples', 'max_depth') else float(best[k])) for k in SPACE}
    model = lgb.LGBMClassifier(n_estimators=int(best['n_estimators']), subsample_freq=1, random_state=SEED,
                               verbose=-1, n_jobs=-1, **params).fit(X_pre, y_pre)
    joblib.dump(model, os.path.join(tu.MODELS, f'lgbm_V1_pre2014_{variant}.pkl'))
    with open(os.path.join(tu.MODELS, f'lgbm_V1_pre2014_{variant}_params.json'), 'w') as f:
        json.dump({'params': params, 'n_estimators': int(best['n_estimators']), 'training_rows': len(d)}, f, indent=2)

    pp = pred[['stay_id', 'hospital_id', 'mortality', 'split']].merge(e[['stay_id'] + FEATS], on='stay_id',
                                                                     how='left', validate='one_to_one')
    if pp[FEATS].isna().all(axis=1).any():
        raise ValueError('Some eICU stays have all features missing after the feature join.')
    pp['p_pre'] = model.predict_proba(pp[FEATS].astype(float))[:, 1]
    pp[['stay_id', 'hospital_id', 'mortality', 'split', 'p_pre']].to_parquet(
        os.path.join(tu.MODELS, f'pred_eicu_pre2014_{variant}.parquet'), index=False)
    PRED_PRE[variant] = pp[['stay_id', 'hospital_id', 'mortality', 'split', 'p_pre']]
    PI_SRC_PRE[variant] = float(y_pre.mean())

    # Pooled performance beside the original model (same eICU stays; MIMIC 2017-2022 test set).
    p_tt = model.predict_proba(tt_m[FEATS].astype(float))[:, 1]
    for where, y_, p_ in [('MIMIC-IV temporal test', tt_m['mortality'].to_numpy(), p_tt),
                          ('eICU-CRD all stays', pp['mortality'].to_numpy(), pp['p_pre'].to_numpy())]:
        r = tu.evaluate(y_, p_, full=False)
        perf_rows.append({'model': f'pre2014_{variant}', 'data': where, **{k: r[k] for k in ['n', 'auroc', 'oe', 'citl', 'slope', 'scaled_brier']}})
r0 = tu.evaluate(pred['mortality'].to_numpy(), pred['p_lgbm'].to_numpy(), full=False)
perf_rows.append({'model': 'original_2008_2016', 'data': 'eICU-CRD all stays', **{k: r0[k] for k in ['n', 'auroc', 'oe', 'citl', 'slope', 'scaled_brier']}})
tu.save_table(pd.DataFrame(perf_rows).round(4), 'D_pre2014_model_performance')


### 5.2 Repeat the target-site comparison with each earlier-window model

Same hospitals and 2014/2015 rows as above. Cohorts are the two outcome-free sets defined from the original model's expected deaths, so the comparison set is held constant across model variants.


In [ ]:
pre_methods = {'No updating': 'S0_none', 'Global logistic': 'S1b_global_logistic',
               'Local intercept': 'S2a_local_intercept', 'Empirical Bayes (DL prior)': 'S3_eb',
               'Prevalence correction (all labels)': 'S6_prev_full_labels'}
cmp_rows, cmp_t5, cmp_wil = [], [], []
for variant, pp in PRED_PRE.items():
    upd_pre = pp[pp['split'] == 'update'].reset_index(drop=True)
    ev_pre = pp[pp['split'] == 'eval'].reset_index(drop=True)
    pre_point, _, _ = tu.run_strategies(upd_pre, ev_pre, pcol='p_pre', ycol='mortality')
    rate_pre = {h: clipped_rate(g['mortality'].to_numpy(float)) for h, g in upd_pre.groupby('hospital_id')}
    r_ = pre_point['hospital_id'].map(rate_pre).astype(float).to_numpy()
    pre_point['S6_prev_full_labels'] = tu.expit(tu.logit(pre_point['p_pre'].to_numpy(float)) + tu.logit(r_)
                                                - tu.logit(PI_SRC_PRE[variant]))
    for cohort_name, hset in [('all_both_years_outcome_free', H_ALL_BOTH), ('outcome_free_volume_expected', H_STRICT)]:
        for method, pcol in pre_methods.items():
            per = per_site_counts(pre_point, pcol, hset)
            cmp_rows.append({'model': f'pre2014_{variant}', 'cohort': cohort_name, 'method': method,
                             **fit_poisson_lognormal(per['observed'], per['expected'])})
            if cohort_name == 'outcome_free_volume_expected':
                cmp_t5.append({'model': f'pre2014_{variant}', 'method': method,
                               **table5_summary(pre_point, pcol, hset, n_sim=N_SIM_BENCH, seed=SEED)})
    cmp_wil.append(wilcoxon_holm_vs_ref(pre_point, pre_methods, 'Empirical Bayes (DL prior)', H_STRICT)
                   .assign(model=f'pre2014_{variant}'))

orig_pln = pln_summary[pln_summary['cohort'].isin(['all_both_years_outcome_free', 'outcome_free_volume_expected'])
                       & pln_summary['method'].isin(pre_methods)].assign(model='original_2008_2016')
combined = pd.concat([orig_pln, pd.DataFrame(cmp_rows)], ignore_index=True, sort=False)
tu.save_table(combined.round(6), 'D_original_vs_pre2014_poisson_lognormal')
orig_t5 = t5[(t5['cohort'] == 'outcome_free_volume_expected') & t5['method'].isin(pre_methods)].drop(columns='cohort').assign(model='original_2008_2016')
tu.save_table(pd.concat([orig_t5, pd.DataFrame(cmp_t5)], ignore_index=True).round(4), 'D_original_vs_pre2014_table5_style')
tu.save_table(pd.concat(cmp_wil, ignore_index=True).round(5), 'D_pre2014_wilcoxon_holm_vs_EB')


## 6. Output checklist and interpretation safeguards

- `A_label_budget_paired_contrasts.csv`: equal-n contrasts (prevalence − EB, EB − none, prevalence − none, local − EB) with hospital-bootstrap CIs.
- `A_full_label_paired_contrasts_bootstrap.csv`: all-label prevalence correction vs all-label EB (unequal-budget setting; intervals conditional on the fitted updates).
- `B_hospital_flow.csv`: denominators for every analysis; confirm no 2015 death count enters the outcome-free selection.
- `B_table5_style_by_cohort.csv` and `B_wilcoxon_holm_vs_EB.csv`: the Table 5 statistics and tests in the outcome-free cohorts, beside the original 97-hospital set.
- `B_poisson_lognormal_summary.csv`: count-model τ including zero-event hospitals; compare methods within this estimator only.
- `C_pipeline_bootstrap_summary.csv`, `C_pipeline_bootstrap_paired_tau.csv`, `C_network_prior_DL_vs_REML_point_estimates.csv`: hospital-cluster bootstrap with the prior re-estimated each replicate; check `share_not_converged` and that the bootstrap medians are close to the point estimates.
- `D_pre2014_cohort_summary.csv`, `D_pre2014_model_performance.csv`, `D_original_vs_pre2014_*.csv`: earlier-window models; the strict variant supports the wording "trained only on stays before 2014", within the limits of date shifting.
- `environment_pip_freeze.txt`, `environment.json`: the environment that produced the run.

NB14 does not rewrite conclusions. Review all outputs before changing the manuscript.


In [ ]:
# Record the exact environment that produced the reported run; do not pin versions by hand.
pip_freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], check=True,
                            capture_output=True, text=True).stdout
with open(os.path.join(OUT, 'environment_pip_freeze.txt'), 'w', encoding='utf-8') as f:
    f.write(pip_freeze)
with open(os.path.join(OUT, 'environment.json'), 'w', encoding='utf-8') as f:
    json.dump({'python_version': sys.version, 'platform': platform.platform(),
               'transport_fast': bool(FAST), 'random_seed': int(SEED), 'notebook_version': 'NB14 v2'}, f, indent=2)

# Final run summary and result archive. Do not interpret this summary as a pass/fail scientific decision.
summary = {
    'label_budget_repeats': int(N_LABEL_REPEATS),
    'label_budget_ci_bootstrap': int(B_LABEL_CI),
    'pipeline_bootstrap_requested': int(B_PIPELINE),
    'pipeline_bootstrap_completed': int(boot_df['rep'].nunique()) if 'boot_df' in globals() and not boot_df.empty else 0,
    'update_hospitals': int(len(UPDH)),
    'hospitals_both_years': int(len(H_BOTH)),
    'original_outcome_selected_hospitals': int(len(ELIG_ORIG)),
    'outcome_free_all_both_years': int(len(H_ALL_BOTH)),
    'outcome_free_volume_expected': int(len(H_STRICT)),
    'fixed35_ge400_update': int(len(FIX35)),
    'pre2014_training_rows': {k: int(len(d)) for k, d in M_PRE.items()},
    'pre2014_source_prevalence': {k: float(v) for k, v in PI_SRC_PRE.items()},
    'fast_mode': bool(FAST),
    'notes': 'Inspect all CSVs and convergence flags. Fast-mode output is for smoke testing only.'
}
zip_path = tu.finish_notebook(summary)
print('NB14 result archive:', zip_path)